# How to train your AI: fra statistiske modeller til nevrale nettverk

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/simeneide/how-to-train-your-ai/blob/main/workshop.ipynb)

Tech Day, Norges Bank, 6. november 2026. Simen Eide.

Vi skal prognosere **KPI-JAE neste måned**. Først med modeller fra SMART (AR, VAR, BVAR, faktormodell), bygget i PyTorch. Så gjør vi dem om til nevrale nettverk.

**Slik bruker du notebooken:** trykk ▶ på cellene ovenfra og ned. Du trenger ikke skrive kode. I hver oppgave endrer du tallene i skjemaet til høyre og kjører cellen på nytt. Vil du kode, er alt synlig.

Alle modeller følger de samme fire stegene som i foredraget, og koden i hver oppgave er delt opp slik:

| Steg | Spørsmål | Her |
|---|---|---|
| **1. Data** | Hva vet vi? | hvilke serier, og hvor mange måneder tilbake |
| **2. Modell** | Hvordan henger det sammen? | en funksjon med parametere: `nn.Linear`, `nn.Sequential` |
| **3. Trening** | Hvilke parametere passer best? | treningsløkka, prior (weight decay), valgt på validering |
| **4. Beslutning** | Hva gjør vi med svaret? | prognosefeil, ledertavle og prognose for oktober |

Bakgrunn om KPI-JAE, dataene og SMART står i README.

In [ ]:
# @title Oppsett: biblioteker og datafil { display-mode: "form" }
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

DATA = Path("data/forTecDay.xlsx")
if (
    not DATA.exists()
):  # Colab: hent fila fra GitHub, eller last den opp så lenge repoet er privat
    import urllib.request

    DATA.parent.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/simeneide/how-to-train-your-ai/main/data/forTecDay.xlsx",
            DATA,
        )
    except OSError:
        from google.colab import files

        DATA = Path(next(iter(files.upload())))

## Steg 1: Data

Vi bygger datasettet én gang her, og alle oppgavene bruker det samme. Oppgavene velger bare hvilke kolonner modellen får se.

Først rådataene: 61 månedlige serier med indeksnivåer, kronekurs, oljepris og ledighet.

In [ ]:
raw = pd.read_excel(DATA).set_index("Time")
raw.index = pd.PeriodIndex([t.replace("M", "-") for t in raw.index], freq="M")
raw = raw.loc["2001-01":]

TARGET = "MSA_PCPIJAE"
DRIVERS = {
    "kronekurs": "I44",
    "oljepris": "DS.OILBRSV",
    "importpriser": "MSA_IPK",
    "innsatsvarer": "MSA_IPI",
    "ledighet": "NAV.HL.TOT.PST.S",
}
GROUPS = [
    c for c in raw if c.startswith(("MSA_PCPIJAE_25", "KPI.KPI"))
]  # 13 hovedgrupper og 40 delindekser
print(f"{raw.shape[1]} serier, {raw.index[0]} til {raw.index[-1]}")
raw[[TARGET, "MSA_PCPIJAE_25_K1", "KPI.KPI.25.K041.IPR.S", *DRIVERS.values()]].tail()

Vi skal prognosere indeksnivået neste måned. Siden vi vet nivået denne måneden, er det samme problem som å prognosere **endringen fra én måned til neste**:

$$\text{nivå}_{t+1} = \text{nivå}_t \times (1 + \text{endring}_{t+1})$$

Vi lar modellene jobbe med endringen, i prosent. Nivået stiger nesten hele tiden, så en modell på nivå lærer mest at «neste måned ligner denne», og 2026-nivåer er større enn noe den har sett i treningsdataene. Endringen svinger rundt et stabilt snitt, så trening, validering og test er på samme skala. Feilen blir også lett å lese: en RMSE på 0,2 betyr at prognosen for nivået bommer med rundt 0,2 % i snitt. Til slutt regner vi prognosen tilbake til nivå og tolvmånedersvekst.

Samme omregning gjør vi for alle prisseriene, kronekursen og oljeprisen. Ledigheten er allerede i prosent, så der bruker vi endringen i prosentpoeng.

Vi deler historien i tre: **trening** (2003-2016) der modellen lærer, **validering** (2017-2020) der vi velger modell, og **test** (2021-) der vi ser hvor godt det gikk. Testperioden har inflasjonssjokket.

In [ ]:
RATES = ["NAV.HL.TOT.PST.S"]
mm = 100 * np.log(raw.drop(columns=RATES)).diff()
mm[RATES] = raw[RATES].diff()

SPLIT = {
    "trening": ("2003-01", "2016-12"),
    "validering": ("2017-01", "2020-12"),
    "test": ("2021-01", "2100-01"),
}
_, ax = plt.subplots(figsize=(14, 3.6))
for (name, (a, b)), c in zip(SPLIT.items(), ["tab:blue", "tab:orange", "tab:red"]):
    s = mm[TARGET].loc[a:b]
    ax.plot(s.index.to_timestamp(), s, color=c, label=name)
ax.set_ylabel("Endring i KPI-JAE, %")
ax.legend()
plt.show()

Nå bygger vi datasettet. Én rad er én måned **t**:

- **y** (fasiten): endringen i KPI-JAE måneden etter, t+1.
- **X** (det modellen vet i måned t): endringen i KPI-JAE de 24 siste månedene, driverne de 3 siste månedene og alle 53 gruppene de 2 siste månedene. `t-0` er denne måneden, `t-1` forrige, osv.

Alle kolonnene standardiseres med gjennomsnitt og standardavvik fra treningsperioden, så de er på samme skala. Tall vi mangler (noen serier starter sent), settes til gjennomsnittet.

In [ ]:
NAMES = {TARGET: "KPI-JAE", **{v: k for k, v in DRIVERS.items()}}
NAMES |= {
    c: c.replace("MSA_PCPIJAE_25_", "gruppe ")
    .replace("KPI.KPI.25.", "delindeks ")
    .replace(".IPR.S", "")
    for c in GROUPS
}
MAX_LAGS = {
    TARGET: 24,
    **dict.fromkeys(DRIVERS.values(), 3),
    **dict.fromkeys(GROUPS, 2),
}

X = pd.concat(
    {
        f"{NAMES[c]} t-{k}": mm[c].shift(k)
        for c, p in MAX_LAGS.items()
        for k in range(p)
    },
    axis=1,
)
y = mm[TARGET].shift(-1).rename("KPI-JAE neste måned")

train_part = X.loc[slice(*SPLIT["trening"])]
Z = ((X - train_part.mean()) / train_part.std().replace(0, 1)).fillna(0.0)
ROWS = {name: y.loc[a:b].dropna().index for name, (a, b) in SPLIT.items()}

print(f"X: {X.shape[0]} måneder x {X.shape[1]} kolonner")
print({name: len(idx) for name, idx in ROWS.items()}, "måneder i hver del")
pd.concat([y, X.iloc[:, [0, 1, 2, 24, 27, 39]]], axis=1).loc["2026"].round(2)

Hver oppgave plukker kolonner fra X med `kolonner(...)`:

In [ ]:
def kolonner(
    kpi: int = 12, drivere: list[str] = (), grupper: bool = False
) -> list[str]:
    """Kolonnene i X: de `kpi` siste månedene av KPI-JAE, valgte drivere, og eventuelt alle gruppene."""
    cols = [f"KPI-JAE t-{k}" for k in range(kpi)]
    cols += [c for d in drivere for c in X if c.startswith(f"{d} t-")]
    if grupper:
        cols += [c for g in GROUPS for c in X if c.startswith(f"{NAMES[g]} t-")]
    return cols


kolonner(kpi=3, drivere=["kronekurs"])

In [ ]:
# @title Maskinrom: treningsløkka, evaluering og ledertavle { display-mode: "form" }
LEADERBOARD: list[dict] = []


def tensors(cols: list[str]) -> dict:
    out = {
        name: (
            torch.tensor(Z.loc[idx, cols].values, dtype=torch.float32),
            torch.tensor(y.loc[idx].values, dtype=torch.float32),
            idx,
        )
        for name, idx in ROWS.items()
    }
    out["siste"] = torch.tensor(
        Z[cols].iloc[[-1]].values, dtype=torch.float32
    )  # siste kjente måned, for prognosen
    return out


def train(
    model: nn.Module,
    cols: list[str],
    epochs: int = 2000,
    lr: float = 0.01,
    weight_decay: float = 0.0,
    seed: int = 0,
):
    """Treningsløkka. Vi tar vare på vektene som traff best på valideringsperioden (early stopping)."""
    torch.manual_seed(seed)
    for m in model.modules():
        if hasattr(m, "reset_parameters"):
            m.reset_parameters()
    data = tensors(cols)
    (xtr, ytr, _), (xva, yva, _) = data["trening"], data["validering"]
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    hist, best, state = {"trening": [], "validering": []}, float("inf"), None
    for _ in range(epochs):
        model.train()
        opt.zero_grad()
        loss = ((model(xtr).squeeze(-1) - ytr) ** 2).mean()
        loss.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            val = ((model(xva).squeeze(-1) - yva) ** 2).mean().item()
        hist["trening"].append(loss.item() ** 0.5)
        hist["validering"].append(val**0.5)
        if val < best:
            best, state = val, {k: v.clone() for k, v in model.state_dict().items()}
    model.load_state_dict(state)
    model.eval()
    return model, data, hist


def evaluate(name: str, model: nn.Module, data: dict, hist: dict | None = None) -> None:
    """RMSE i hver periode, en graf av prognosene i testperioden, og en rad på ledertavla."""
    res = {"modell": name, "parametere": sum(p.numel() for p in model.parameters())}
    with torch.no_grad():
        for part in ["trening", "validering", "test"]:
            x, yt, idx = data[part]
            res[part] = ((model(x).squeeze(-1) - yt) ** 2).mean().item() ** 0.5
        res["prognose neste mnd"] = model(data["siste"]).item()
        x, yt, idx = data["test"]
        p = model(x).squeeze(-1).numpy()
    LEADERBOARD.append(res)
    _, axes = plt.subplots(1, 2 if hist else 1, figsize=(14 if hist else 9, 3.6))
    ax = axes[0] if hist else axes
    t = idx.to_timestamp()
    ax.plot(t, yt.numpy(), color="black", label="Faktisk")
    ax.plot(t, p, color="tab:red", label=name)
    ax.set_title(f"Testperioden: RMSE {res['test']:.3f}")
    ax.set_ylabel("Endring i KPI-JAE, %")
    ax.legend()
    if hist:
        axes[1].plot(hist["trening"], label="Trening")
        axes[1].plot(hist["validering"], label="Validering")
        axes[1].set_title("Treningsløkka: feil per runde")
        axes[1].set_xlabel("Runde")
        axes[1].set_ylabel("RMSE")
        axes[1].set_ylim(0, max(0.6, min(hist["validering"]) * 2))
        axes[1].legend()
    plt.tight_layout()
    plt.show()
    print({k: round(v, 3) if isinstance(v, float) else v for k, v in res.items()})


def leaderboard() -> pd.DataFrame:
    return (
        pd.DataFrame(LEADERBOARD)
        .drop_duplicates("modell", keep="last")
        .sort_values("validering")
        .round(3)
        .reset_index(drop=True)
    )


# Enkle referanser uten parametere
for name, p in {
    "Naiv: samme som forrige måned": mm[TARGET],
    "Snitt siste 12 mnd": mm[TARGET].rolling(12).mean(),
}.items():
    LEADERBOARD.append(
        {
            "modell": name,
            "parametere": 0,
            **{
                k: float(np.sqrt(((p.loc[idx] - y.loc[idx]) ** 2).mean()))
                for k, idx in ROWS.items()
            },
            "prognose neste mnd": float(p.iloc[-1]),
        }
    )
leaderboard()

## Oppgave 1: AR-modellen er et nevralt nett med ett lag

SMART har mange AR-modeller: neste måned er en vektet sum av de `P` siste månedene. I PyTorch er det `nn.Linear(P, 1)`: `P` noder inn, én ut, og vektene er strekene mellom dem.

**Prøv:** endre `P` (antall måneder modellen ser tilbake). Hva skjer med testfeilen? Hvilke måneder får mest vekt?

In [ ]:
# 1. Data: de P siste månedene av KPI-JAE
P = 12  # @param {type:"slider", min:1, max:24, step:1}
cols_ar = kolonner(kpi=P)

# 2. Modell: én vekt per måned, pluss et konstantledd
ar = nn.Linear(P, 1)

# 3. Trening
EPOCHS = 2000  # @param {type:"integer"}
LR = 0.01  # @param {type:"number"}
ar, data, hist = train(ar, cols_ar, epochs=EPOCHS, lr=LR)

# 4. Beslutning: hvor godt traff den, og hva sier den om neste måned?
evaluate(f"AR({P})", ar, data, hist)

# Samme svar som minste kvadraters metode (OLS)?
xtr, ytr, _ = data["trening"]
ols = torch.linalg.lstsq(
    torch.cat([xtr, torch.ones(len(xtr), 1)], 1), ytr.unsqueeze(1)
).solution.squeeze()
plt.figure(figsize=(9, 3))
plt.bar(
    np.arange(1, P + 1) - 0.2,
    ar.weight.detach().squeeze().numpy(),
    width=0.4,
    label="PyTorch",
)
plt.bar(np.arange(1, P + 1) + 0.2, ols[:-1].numpy(), width=0.4, label="OLS")
plt.xlabel("Måneder tilbake")
plt.ylabel("Vekt")
plt.title("Vektene i AR-modellen")
plt.legend()
plt.show()

## Oppgave 2: VAR og en prior

En VAR-likning tar med andre serier: kronekurs, olje, importpriser, ledighet. Med `MED_GRUPPER` tar vi også med 53 delindekser av KPI. Da får modellen flere parametere enn vi har måneder å trene på.

**Prøv:**
1. Skru på `MED_GRUPPER` med `WEIGHT_DECAY = 0`. Se på grafen til høyre: treningsfeilen går ned, valideringsfeilen går opp. Det er **overtilpasning**.
2. Øk `WEIGHT_DECAY`. Det er det samme som en Bayesiansk prior (som i BVAR): vektene trekkes mot null. Når blir testfeilen best?

In [ ]:
# 1. Data: KPI-JAE pluss drivere, og eventuelt alle delindeksene
kronekurs = True  # @param {type:"boolean"}
oljepris = True  # @param {type:"boolean"}
importpriser = True  # @param {type:"boolean"}
innsatsvarer = False  # @param {type:"boolean"}
ledighet = False  # @param {type:"boolean"}
MED_GRUPPER = False  # @param {type:"boolean"}

chosen = [
    k
    for k, v in {
        "kronekurs": kronekurs,
        "oljepris": oljepris,
        "importpriser": importpriser,
        "innsatsvarer": innsatsvarer,
        "ledighet": ledighet,
    }.items()
    if v
]
cols_var = kolonner(kpi=12, drivere=chosen, grupper=MED_GRUPPER)
print(f"{len(cols_var)} kolonner, {len(ROWS['trening'])} måneder å trene på")
# 2. Modell: fortsatt én vekt per input
var = nn.Linear(len(cols_var), 1)

# 3. Trening: weight decay er prioren som trekker vektene mot null
WEIGHT_DECAY = 0.0  # @param [0.0, 0.01, 0.1, 0.3, 1.0, 3.0] {type:"raw"}
var, data, hist = train(var, cols_var, weight_decay=WEIGHT_DECAY)

# 4. Beslutning
evaluate(
    f"VAR ({', '.join(chosen) or 'ingen drivere'}{', grupper' if MED_GRUPPER else ''}, wd={WEIGHT_DECAY})",
    var,
    data,
    hist,
)

## Oppgave 3: fra faktormodell til nevralt nett

En faktormodell komprimerer mange serier til noen få **faktorer**, og prognoserer fra dem. I PyTorch er det to lag: `Linear(n, R)` og så `Linear(R, 1)`. Det midterste laget er et **skjult lag**, en flaskehals.

Legg til en ikke-lineær funksjon (ReLU) i det skjulte laget, og vi har et ekte nevralt nettverk.

**Prøv:** endre antall faktorer `R`, bytt `AKTIVERING` til ReLU, og legg til `DROPOUT` (skru av tilfeldige noder under trening, en annen type regularisering).

In [ ]:
# 1. Data: alt vi har
cols_big = kolonner(kpi=12, drivere=list(DRIVERS), grupper=True)

# 2. Modell: input -> R faktorer (skjult lag) -> prognose
R = 3  # @param {type:"slider", min:1, max:64, step:1}
AKTIVERING = "ingen (faktormodell)"  # @param ["ingen (faktormodell)", "ReLU", "Tanh"]
DROPOUT = 0.0  # @param {type:"slider", min:0, max:0.8, step:0.1}
act = {"ingen (faktormodell)": nn.Identity(), "ReLU": nn.ReLU(), "Tanh": nn.Tanh()}[
    AKTIVERING
]
net = nn.Sequential(
    nn.Linear(len(cols_big), R), act, nn.Dropout(DROPOUT), nn.Linear(R, 1)
)
print(net)

# 3. Trening
WEIGHT_DECAY_3 = 0.1  # @param [0.0, 0.01, 0.1, 0.3, 1.0, 3.0] {type:"raw"}
net, data, hist = train(net, cols_big, lr=0.003, weight_decay=WEIGHT_DECAY_3)

# 4. Beslutning
evaluate(
    f"{AKTIVERING}, R={R}, dropout={DROPOUT}, wd={WEIGHT_DECAY_3}", net, data, hist
)

## Oppgave 4: slå AR

Ingen av modellene over slår AR-modellen lett. Kan du? Bygg din egen modell under. Alt er lov, men velg modell på **valideringsfeilen**, ikke testfeilen. Ellers jukser du med framtiden.

Idéer: færre eller flere lag, bare noen av gruppene, kortere historikk (`p_target`), flere skjulte lag, kombiner prognosene fra flere modeller (som SMART gjør).

In [ ]:
NAVN = "Min modell"  # @param {type:"string"}

# 1. Data
cols_mine = kolonner(kpi=6, drivere=["kronekurs", "importpriser"])

# 2. Modell
mine = nn.Sequential(
    nn.Linear(len(cols_mine), 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)
# 3. Trening
mine, data, hist = train(mine, cols_mine, lr=0.003, weight_decay=0.3)

# 4. Beslutning
evaluate(NAVN, mine, data, hist)

## Ledertavla

Sortert etter valideringsfeil. Den som er best på validering, er den vi ville valgt. Stemmer det med test?

In [ ]:
leaderboard()

## Steg 4: Beslutning. Hva blir KPI-JAE i oktober?

Modellen som var best på validering, gir en prognose for oktober. SSB publiserer tallet 10. november.

In [ ]:
lb = leaderboard()
best = lb.iloc[0]
level = raw[TARGET]
oct_level = level.iloc[-1] * np.exp(best["prognose neste mnd"] / 100)
yoy = 100 * (oct_level / level.loc[str(level.index[-1] - 11)] - 1)
print(f"Beste modell på validering: {best['modell']}")
print(
    f"Prognose for {level.index[-1] + 1}: {best['prognose neste mnd']:+.2f} % fra måneden før, {yoy:.1f} % fra samme måned i fjor (sesongjustert)"
)